# Imports

In [1]:
import optuna

import numpy as np
import pandas as pd

from utils import load_pickle

from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
to_drop = ['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0', 'xgb_1', 'xgb_2']

X_train = pd.read_parquet('../data/X_train_stacking_layer_two.parquet') #.drop(to_drop, axis=1)
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_two.parquet') #.drop(to_drop, axis=1)

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.999910,0.000083,0.000007,0.999866,0.000124,0.000010,0.999928,0.000070,0.000003,0.988210,0.004367,0.007423,0.999210,0.000790,0.000000
1,0.992961,0.000411,0.006628,0.991422,0.000567,0.008011,0.992562,0.000566,0.006872,0.986847,0.004779,0.008373,0.988522,0.001245,0.010233
2,0.000109,0.999858,0.000033,0.000162,0.999819,0.000019,0.000030,0.999962,0.000008,0.009897,0.988895,0.001208,0.002604,0.997396,0.000000
3,0.999805,0.000187,0.000009,0.999766,0.000223,0.000010,0.999837,0.000159,0.000004,0.988202,0.004372,0.007426,0.999098,0.000902,0.000000
4,0.998393,0.001560,0.000047,0.998544,0.001413,0.000043,0.998238,0.001711,0.000051,0.987966,0.004405,0.007629,0.998207,0.001793,0.000000


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.997841,0.002113,0.000045,0.997856,0.001945,0.000199,0.998030,0.001934,0.000036,0.987950,0.004507,0.007543,0.997783,0.002217,0.000000
1,0.996887,0.003080,0.000033,0.997599,0.002381,0.000020,0.996744,0.003240,0.000016,0.987886,0.004549,0.007565,0.997324,0.002676,0.000000
2,0.998244,0.001026,0.000731,0.998669,0.000563,0.000768,0.998349,0.000750,0.000901,0.987787,0.004504,0.007709,0.998949,0.001051,0.000000
3,0.000658,0.000086,0.999256,0.001490,0.000130,0.998380,0.000538,0.000138,0.999324,0.013634,0.004166,0.982199,0.000000,0.002127,0.997873
4,0.999840,0.000151,0.000009,0.999810,0.000178,0.000012,0.999880,0.000116,0.000004,0.988019,0.004456,0.007524,0.999069,0.000931,0.000000


# Machine Learning

In [ ]:
def objective(trial, X, y):
    
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    C = trial.suggest_float("C", 1e-5, 100.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        model = LogisticRegression(
            solver="saga",
            C=C,
            l1_ratio=l1_ratio,
            class_weight=class_weight,
            fit_intercept=fit_intercept,
            tol=tol,
            max_iter=max_iter,
            random_state=42,
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)
        
        weighted_probas = proba * weights
        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.class_encoded), 
    n_trials=5, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-28 16:26:31,970] A new study created in memory with name: no-name-b1190b1a-5306-4bb2-86dd-502d8a4cccaf
  0%|          | 0/90 [00:00<?, ?it/s]

In [21]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=80, n_jobs=-1, show_progress_bar=True)

print("\nBest trial score:")
print(study.best_trial.value)
print("\nBest params:")
print(study.best_trial.params)

Best trial: 75. Best value: 0.966258:   1%|█▌                                                                                                                           | 1/80 [00:15<20:52, 15.85s/it]

[I 2026-06-26 15:45:08,405] Trial 122 pruned. 


Best trial: 75. Best value: 0.966258:   2%|███▏                                                                                                                         | 2/80 [00:24<15:00, 11.55s/it]

[I 2026-06-26 15:45:16,939] Trial 131 finished with value: 0.9658178227633112 and parameters: {'l1_ratio': 0.20654963379626076, 'C': 0.09400379967483277, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.0030652450026953104, 'max_iter': 4708, 'weight_class_0': 2.1348314938374613, 'weight_class_1': 1.7975410138930061, 'weight_class_2': 1.9188438745228857}. Best is trial 75 with value: 0.9662583810460816.


Best trial: 75. Best value: 0.966258:   4%|████▋                                                                                                                        | 3/80 [00:24<08:18,  6.48s/it]

...

[I 2026-06-26 15:48:17,251] Trial 197 finished with value: 0.9660550066799594 and parameters: {'l1_ratio': 0.8249705669229007, 'C': 0.07136309590840828, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 9.600312478944951e-05, 'max_iter': 4231, 'weight_class_0': 1.2203818627208338, 'weight_class_1': 2.7171603453135122, 'weight_class_2': 4.07843678680907}. Best is trial 193 with value: 0.9663482802797547.


Best trial: 193. Best value: 0.966348:  99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍ | 79/80 [03:26<00:02,  2.22s/it]

[I 2026-06-26 15:48:19,097] Trial 198 finished with value: 0.9657278861858953 and parameters: {'l1_ratio': 0.83164015645327, 'C': 0.0018499381248313755, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 9.585512029126983e-05, 'max_iter': 4090, 'weight_class_0': 1.212311396828259, 'weight_class_1': 2.001532621714819, 'weight_class_2': 4.120032289953299}. Best is trial 193 with value: 0.9663482802797547.


Best trial: 193. Best value: 0.966348: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [03:37<00:00,  2.71s/it]

[I 2026-06-26 15:48:29,570] Trial 199 finished with value: 0.9660674237420359 and parameters: {'l1_ratio': 0.9353065668405115, 'C': 0.06655056548619691, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00010611153678190407, 'max_iter': 4050, 'weight_class_0': 1.3644101139063267, 'weight_class_1': 2.664329469212658, 'weight_class_2': 4.268092894188272}. Best is trial 193 with value: 0.9663482802797547.

Best trial score:
0.9663482802797547

Best params:
{'l1_ratio': 0.33444640323378966, 'C': 0.009816711499908143, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 9.402666379694602e-05, 'max_iter': 4393, 'weight_class_0': 1.3891686022004115, 'weight_class_1': 2.802558290832105, 'weight_class_2': 4.261825099062885}


In [28]:
lg_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(
    solver="saga",
    random_state=42,
    **lg_params
).fit(X_train, y_train.class_encoded)

test_proba = lg.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [29]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [30]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lg.csv', index=False)

In [31]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [32]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'lg_0', 'lg_1', 'lg_2', 'sgd_0', 'sgd_1', 'sgd_2'],
      dtype='str')

In [33]:
len(study.trials)

200